# 00 · Foundations of Probabilistic Forecasting

**Goal of this notebook:** build the vocabulary that every later notebook relies on.

| # | Notebook | Forecast type |
|---|----------|---------------|
| 00 | Foundations (this one) | concepts, data, scoring rules |
| 01 | Full predictive distribution | PDF / CDF |
| 02 | Quantile forecasts | P10 / P50 / P90 … |
| 03 | Prediction intervals | [lower, upper] with coverage |
| 04 | Conformal prediction | distribution-free intervals |
| 05 | Ensembles & sample paths | Monte-Carlo trajectories |
| 06 | Event probabilities (binary) | P(event) |
| 07 | Categorical & binned (histogram) forecasts | P(class), P(bin) |
| 08 | Distribution families & count data | parametric choices, NegBin, intermittent demand |
| 09 | Multivariate / joint forecasts | copulas, energy score |
| 10 | Hierarchical (coherent) forecasts | reconciliation |
| 11 | Bayesian forecasting | posterior predictive, GP |
| 12 | Evaluation, backtesting & decisions | putting it all together |

All datasets are **monthly**.

---
## 1. Point forecast vs probabilistic forecast

A **point forecast** answers *"what is the single best guess for $y_{T+h}$?"*
A **probabilistic forecast** answers *"what is the full range of plausible values for $y_{T+h}$, and how likely is each?"*

Formally a probabilistic forecast is a **predictive distribution**

$$\hat F_{T+h|T}(y) = \Pr\left(Y_{T+h} \le y \mid \mathcal{I}_T\right)$$

where $\mathcal{I}_T$ is the information available at forecast origin $T$ and $h$ is the horizon.
Every "type" of probabilistic forecast is just a different **summary** of (or a different way to represent) $\hat F$:

| Representation | What you store | Information content |
|---|---|---|
| Full distribution | parameters of $\hat F$ (e.g. $\mu,\sigma$) or the CDF itself | complete |
| Samples / ensemble | $x^{(1)},\dots,x^{(M)} \sim \hat F$ | complete (as $M\to\infty$), and keeps **joint** structure |
| Quantiles | $\hat F^{-1}(\tau)$ for a few $\tau$ | partial |
| Interval | $[\hat F^{-1}(\alpha/2), \hat F^{-1}(1-\alpha/2)]$ | two quantiles |
| Event probability | $1-\hat F(c)$ for a threshold $c$ | one number |
| Categorical / binned | $\hat F(b_k)-\hat F(b_{k-1})$ | discretised CDF |

**Why bother?** Decisions depend on *risk*, not only on the expected value: inventory (stock-outs vs holding cost),
capacity planning, staffing, finance (VaR), energy (reserve margins). Notebook 12 shows that the optimal order quantity
in the newsvendor problem is a *quantile*, not the mean.

## 2. Two sources of uncertainty

* **Aleatoric** (irreducible noise): even a perfect model cannot predict next month's weather-driven demand exactly.
* **Epistemic** (model / parameter uncertainty): we estimated parameters from finite data and may have the wrong model.
  It shrinks with more data. Bayesian methods (nb 11) and ensembles (nb 05) are explicit ways to include it.

A common bug: intervals built from the residual standard deviation only ignore epistemic uncertainty and are **too narrow**.

## 3. Calibration and sharpness

Gneiting, Balabdaoui & Raftery (2007): *"maximise the sharpness of the predictive distribution subject to calibration."*

* **Calibration (reliability)**: statistical consistency between forecasts and observations.
  80% intervals should contain ~80% of outcomes; 30% rain forecasts should verify ~30% of the time.
* **Sharpness**: concentration of the predictive distribution, a property of the forecast **alone**.
  A climatological forecast (same wide distribution every month) is calibrated but not sharp, so it is useless for decisions.

## 4. The Probability Integral Transform (PIT)

If $Y \sim F$ (continuous) then $U = F(Y) \sim \text{Uniform}(0,1)$.
So, if our forecasts are calibrated, the values $u_t = \hat F_t(y_t)$ should look uniform.

| PIT histogram shape | Diagnosis |
|---|---|
| flat | calibrated |
| U-shaped | forecasts **too narrow** (over-confident) |
| ∩-shaped (hump) | forecasts **too wide** (under-confident) |
| sloped | **biased** (systematically too high or too low) |

## 5. Proper scoring rules

A scoring rule $S(\hat F, y)$ assigns a penalty to forecast $\hat F$ when $y$ occurs.
It is **(strictly) proper** if the expected score is minimised (uniquely) by reporting your true belief:
$\mathbb{E}_{Y\sim G}[S(G,Y)] \le \mathbb{E}_{Y\sim G}[S(F,Y)]$ for all $F$.
Propriety means **you cannot game the metric** by reporting something other than what you believe.

| Score | For | Formula (lower = better) |
|---|---|---|
| Log score / NLL | densities | $-\log \hat f(y)$ |
| CRPS | any univariate distribution | $\int (\hat F(z) - \mathbb 1\{y\le z\})^2 dz = \mathbb E\lvert X-y\rvert - \tfrac12\mathbb E\lvert X-X'\rvert$ |
| Pinball loss | a single quantile $\tau$ | $\rho_\tau(y-q)=\max(\tau(y-q),(\tau-1)(y-q))$ |
| Interval (Winkler) score | central interval | width + $\tfrac{2}{\alpha}$ × distance outside |
| Brier score | binary event | $(p-o)^2$ |
| Ranked probability score | ordinal categories | $\sum_k (P_k - O_k)^2$ on cumulative probs |
| Energy score | multivariate | $\mathbb E\lVert X-y\rVert - \tfrac12\mathbb E\lVert X-X'\rVert$ |
| Variogram score | multivariate (dependence) | see nb 09 |

**Key facts:** CRPS has the same units as $y$ and reduces to MAE for a point forecast. CRPS = $2\int_0^1$ pinball$_\tau\,d\tau$.

**Not proper:** coverage alone (a 100% wide interval always covers), MAPE of the median, "sharpness alone".

The rest of this notebook makes these ideas concrete.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.graph_objects as go
from scipy import stats
import pf_utils as pf
pf.set_style()
rng = np.random.default_rng(0)

## 6. The datasets (all monthly)

1. **AirPassengers** (1949–1960): trend + multiplicative seasonality. The classic Box-Jenkins series.
2. **Synthetic retail hierarchy** (2015–2024): Total → 2 regions → 4 stores, with correlated shocks. Used for
   backtests, multivariate and hierarchical notebooks.
3. **Synthetic SKU demand**: intermittent count data (lots of zeros) for count distributions.

In [2]:
air = pf.load_air_passengers()
retail = pf.make_retail_hierarchy()
sku = pf.make_sku_demand()

fig = pf.subplots(1, 3, titles=["AirPassengers (thousands)", "Retail sales by store", "SKU demand (units, count data)"], height=380)
fig.add_trace(go.Scatter(x=air.index, y=air, name="AirPassengers", line=dict(color=pf.INK["secondary"], width=2),
                         showlegend=False), row=1, col=1)
for i, s in enumerate(pf.BOTTOM_NAMES):                      # fixed slot order: store i always gets SERIES[i]
    fig.add_trace(go.Scatter(x=retail.index, y=retail[s], name=s, line=dict(color=pf.SERIES[i], width=1.5)), row=1, col=2)
fig.add_trace(go.Bar(x=sku.index, y=sku, name="SKU units", marker_color=pf.SERIES[0], showlegend=False), row=1, col=3)
fig.update_layout(hovermode="x unified")
fig.show()
retail.head()

,Total,North,South,North_A,North_B,South_A,South_B
2015-01-01,1245.0,759.9,485.1,486.2,273.7,313.8,171.3
2015-02-01,1334.3,828.8,505.5,527.4,301.4,323.0,182.5
2015-03-01,1550.1,968.5,581.6,629.8,338.7,397.0,184.6
2015-04-01,1734.7,1064.1,670.6,666.1,398.0,442.0,228.6
2015-05-01,1649.8,910.1,739.7,570.2,339.9,470.6,269.1


### Variance stabilisation
AirPassengers' seasonal swings grow with the level (multiplicative). Modelling $\log y$ makes the noise roughly
constant, so Gaussian errors become reasonable. After forecasting on the log scale we **back-transform**:
if $\log Y \sim N(\mu,\sigma^2)$ then $Y$ is **log-normal**, and

* the median of $Y$ is $e^{\mu}$
* the mean of $Y$ is $e^{\mu+\sigma^2/2}$ (the back-transformed mean needs a bias adjustment)
* quantiles transform directly: $Q_Y(\tau)=e^{Q_{\log Y}(\tau)}$ (quantiles are invariant to monotone transforms, means are not)

In [3]:
fig = pf.subplots(1, 2, titles=["Within-year std of y: grows with the level", "Within-year std of log y: roughly constant"], height=340)
for c, s in enumerate([air, np.log(air)], start=1):
    sd = s.groupby(s.index.year).std()
    fig.add_trace(go.Scatter(x=sd.index, y=sd.values, mode="lines+markers", line=dict(color=pf.SERIES[0], width=2),
                             marker=dict(size=8), showlegend=False, hovertemplate="%{x}: %{y:.3f}<extra></extra>"), row=1, col=c)
fig.update_xaxes(title_text="year")
fig.show()

## 7. Different losses elicit different summaries of the distribution

A deep but very practical fact: **the loss function you train/evaluate with decides which functional of $\hat F$ you get.**

| Loss | Optimal point forecast |
|---|---|
| squared error | mean |
| absolute error | median |
| pinball loss at $\tau$ | $\tau$-quantile |
| 0/1 loss on a threshold | most likely side of the threshold |

Let's verify numerically on a skewed (log-normal) distribution.

In [4]:
draws = rng.lognormal(mean=0, sigma=0.8, size=200_000)
grid = np.linspace(0.2, 3, 400)
mse = [np.mean((draws - g) ** 2) for g in grid]
mae = [np.mean(np.abs(draws - g)) for g in grid]
pin90 = [np.mean(pf.pinball_loss(draws, g, 0.9)) for g in grid]

fig = pf.subplots(1, 3, titles=["squared error → mean", "absolute error → median", "pinball τ=0.9 → 90% quantile"], height=380)
for c, (loss, target) in enumerate(zip([mse, mae, pin90], [draws.mean(), np.median(draws), np.quantile(draws, 0.9)]), start=1):
    best = grid[np.argmin(loss)]
    fig.add_trace(go.Scatter(x=grid, y=loss, mode="lines", line=dict(color=pf.SERIES[0], width=2), name="expected loss",
                             showlegend=c == 1, hovertemplate="guess %{x:.2f} → loss %{y:.3f}<extra></extra>"), row=1, col=c)
    pf.add_vline(fig, best, min(loss), max(loss), "argmin of the loss", color=pf.SERIES[1], dash="solid", row=1, col=c, showlegend=c == 1)
    pf.add_vline(fig, target, min(loss), max(loss), "true functional", color=pf.INK["primary"], dash="dot", row=1, col=c, showlegend=c == 1)
    fig.layout.annotations[c - 1].text += f"<br><sup>argmin {best:.2f} · truth {target:.2f}</sup>"
fig.update_xaxes(title_text="point forecast")
fig.update_layout(margin=dict(t=90))
fig.show()

## 8. Calibration vs sharpness, demonstrated with the PIT

We simulate a truth $Y_t \sim N(\mu_t, 1)$ and compare four forecasters:

* **ideal**: $N(\mu_t,1)$
* **over-confident**: $N(\mu_t, 0.5^2)$, too narrow
* **under-confident**: $N(\mu_t, 2^2)$, too wide
* **biased**: $N(\mu_t+0.7, 1)$
* **climatological**: $N(\bar\mu, \text{var}(Y))$, which is calibrated but **not sharp**

In [5]:
n = 5000
mu_t = rng.normal(0, 2, n)
y = rng.normal(mu_t, 1)
forecasters = {
    "ideal": (mu_t, np.ones(n)),
    "over-confident": (mu_t, 0.5 * np.ones(n)),
    "under-confident": (mu_t, 2 * np.ones(n)),
    "biased": (mu_t + 0.7, np.ones(n)),
    "climatological": (np.full(n, y.mean()), np.full(n, y.std())),
}
fig = pf.subplots(1, 5, titles=list(forecasters), shared_yaxes=True, height=330,
                  title="PIT histograms (dashed line = ideal uniform density)")
rows = []
for c, (name, (m, s)) in enumerate(forecasters.items(), start=1):
    pit = stats.norm.cdf(y, m, s)
    pf.add_pit(fig, pit, row=1, col=c)
    rows.append({"forecaster": name,
                 "CRPS": pf.crps_gaussian(y, m, s).mean(),
                 "log score": pf.log_score_gaussian(y, m, s).mean(),
                 "90% coverage": pf.coverage(y, stats.norm.ppf(0.05, m, s), stats.norm.ppf(0.95, m, s)),
                 "sharpness (mean 90% width)": np.mean(stats.norm.ppf(0.95, m, s) - stats.norm.ppf(0.05, m, s))})
fig.update_layout(margin=dict(t=100))
fig.show()
pd.DataFrame(rows).set_index("forecaster")

,CRPS,log score,90% coverage,sharpness (mean 90% width)
forecaster,,,,
ideal,0.560,1.410,0.908,3.290
over-confident,0.605,2.191,0.589,1.645
under-confident,0.653,1.735,0.999,6.579
biased,0.695,1.658,0.813,3.290
climatological,1.267,2.228,0.899,7.388


**Read the table carefully:**
* The **climatological** forecaster has perfect coverage and a flat PIT. Calibration alone is *not enough*;
  its CRPS is much worse because it is not sharp.
* The **ideal** forecaster wins on both proper scores (CRPS and log score), which is what propriety guarantees.
* The log score punishes over-confidence much more harshly than CRPS (it is unbounded when $\hat f(y)\to 0$).
  Use CRPS when robustness matters and the log score when tail calibration matters.

## 9. CRPS intuition

CRPS is the area between the forecast CDF and the step function of the observation:

In [6]:
yobs, m, s = 1.2, 0.0, 1.0
z = np.linspace(-4, 4, 800)
F = stats.norm.cdf(z, m, s); H = (z >= yobs).astype(float)
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.r_[z, z[::-1]], y=np.r_[F, H[::-1]], fill="toself", fillcolor=pf.rgba(pf.SERIES[1], 0.3),
                         line=dict(width=0), hoverinfo="skip", name="gap between the curves (CRPS = ∫ gap² dz)"))
fig.add_trace(go.Scatter(x=z, y=F, name="forecast CDF F(z)", line=dict(color=pf.SERIES[0], width=2),
                         hovertemplate="F(%{x:.2f}) = %{y:.3f}<extra></extra>"))
fig.add_trace(go.Scatter(x=z, y=H, name="observation step 1{y ≤ z}", line=dict(color=pf.INK["primary"], width=2, shape="hv"),
                         hoverinfo="skip"))
fig.update_layout(title=f"CRPS = {pf.crps_gaussian(yobs, m, s):.3f}   (MAE of the mean would be {abs(yobs - m):.3f})",
                  height=380, xaxis_title="z", yaxis_title="cumulative probability")
fig.show()

# check the three equivalent CRPS formulas
samples = rng.normal(m, s, 20_000)
taus = np.linspace(0.005, 0.995, 199)
print("closed form       :", pf.crps_gaussian(yobs, m, s))
print("from samples      :", pf.crps_sample(yobs, samples))
print("from 199 quantiles:", pf.crps_quantiles(yobs, {t: stats.norm.ppf(t, m, s) for t in taus}))

closed form       : 0.7480153178865699
from samples      : 0.7437453765748996
from 199 quantiles: 0.751727485533066


## 10. Backtesting with rolling origins

A single train/test split gives very noisy probabilistic scores (one year = 12 observations!).
**Rolling-origin evaluation** (time-series cross-validation) refits/re-forecasts at many origins:

```
origin 1: [train..........] [h1 h2 h3]
origin 2: [train...........] [h1 h2 h3]
origin 3: [train............] [h1 h2 h3]
```

Scores are then averaged per horizon. Notebooks 03, 06, 07 and 12 use this.

Rules of thumb:
* Never let information from after the origin leak into features, scaling or model selection.
* Report scores **per horizon**: uncertainty grows with $h$, and so should interval width.
* Use **skill scores** relative to a simple benchmark: $\text{Skill} = 1 - S_{\text{model}}/S_{\text{benchmark}}$.
  Seasonal-naive with empirical errors is a strong monthly benchmark.

## 11. What the "types" list is often missing

These topics are in the later notebooks because they matter in practice:
* **Conformal prediction** (nb 04): distribution-free coverage guarantees around *any* model.
* **Sample paths vs marginals** (nb 05): you need paths to forecast quantities like *total demand over the next quarter*.
* **Linear pooling vs quantile averaging** when combining models (nb 05).
* **Recalibration** (isotonic regression, nb 06).
* **Randomised PIT** for discrete data (nb 08).
* **Copulas / Schaake shuffle** for dependence (nb 09).
* **Probabilistic reconciliation** (nb 10).
* **Epistemic vs aleatoric decomposition** (nb 11).
* **Decision-making from quantiles**: newsvendor, safety stock (nb 12).